In [ ]:
class Node:
    def __init__(self, is_leaf=False):
        self.is_leaf = is_leaf
        self.keys = []
        self.children = []
        self.next_sibling = None

class BPlusTree:
    def __init__(self, order=4, depth=3):
        self.root = Node(is_leaf=True)
        self.order = order
        self.depth = depth

    def hash_function(self, name: str) -> int:
        groups = [
            "аб",
            "вг",
            "ґдеє",
            "жзиі",
            "їйкл",
            "мнопр",
            "стуф",
            "хцчш",
            "щьюя"
        ]

        pair_map = {}
        group_digit_map = {}
        for g_idx, letters in enumerate(groups, start=1):
            for pos_idx, ch in enumerate(letters, start=1):
                pair_map[ch] = f"{g_idx}{pos_idx}"
                group_digit_map[ch] = str(g_idx)

        name_clean = "".join(ch for ch in name.lower() if ch.isalpha())

        first = ""
        for i in range(3):
            if i < len(name_clean):
                first += pair_map.get(name_clean[i], "00")
            else:
                first += "00"

        middle_source = name_clean[3:7]
        middle = "".join(group_digit_map.get(ch, "0") for ch in middle_source)
        middle = middle.ljust(4, "0")

        length = str(min(len(name_clean), 99)).zfill(2)
        final_hash_str = first + middle + length

        return int(final_hash_str)

    def height(self):
        h = 1
        node = self.root
        while not node.is_leaf:
            node = node.children[0]
            h += 1
        return h

    def insert(self, name: str):
        key = self.hash_function(name)

        split_result = self.insert_helper(self.root, key, name)

        if split_result is not None:
            if self.height() >= self.depth:
                raise OverflowError(f"Досягнуто максимальну глибину дерева: {self.depth}")

            promoted_key, left_node, right_node = split_result

            new_root = Node(is_leaf=False)
            new_root.keys = [promoted_key]
            new_root.children = [left_node, right_node]

            self.root = new_root

    def split(self, node, key=None, name=None):
        mid = self.order // 2

        if node.is_leaf:
            if key is not None and name is not None:
                node.keys.append((key, name))
                node.keys.sort(key=lambda x: (x[0], x[1]))

            right_node = Node(is_leaf=True)
            right_node.keys = node.keys[mid:]
            node.keys = node.keys[:mid]

            right_node.next_sibling = node.next_sibling
            node.next_sibling = right_node

            parent_key = right_node.keys[0][0]
            return parent_key, node, right_node

        right_node = Node(is_leaf=False)

        promoted_key = node.keys[mid]
        right_node.keys = node.keys[mid + 1:]
        right_node.children = node.children[mid + 1:]

        node.keys = node.keys[:mid]
        node.children = node.children[:mid + 1]

        return promoted_key, node, right_node

    def insert_helper(self, node, key, name):
        if node.is_leaf:
            if len(node.keys) == self.order - 1:
                return self.split(node, key, name)

            node.keys.append((key, name))
            node.keys.sort(key=lambda x: (x[0], x[1]))
            return None

        i = 0
        while i < len(node.keys) and key >= node.keys[i]:
            i += 1

        split_return = self.insert_helper(node.children[i], key, name)

        if split_return is None:
            return None

        promoted_key, left_node, right_node = split_return

        node.keys.insert(i, promoted_key)
        node.children[i] = left_node
        node.children.insert(i + 1, right_node)

        if len(node.keys) == self.order:
            return self.split(node)

        return None

    def find_leaf_entry(self, key, name):
        leaf = self.get_first_leaf()
        while leaf:
            for i, (k, v) in enumerate(leaf.keys):
                if k == key and v == name:
                    return leaf, i
            leaf = leaf.next_sibling
        return None, None

    def search_exact(self, name: str):
        key = self.hash_function(name)
        leaf, idx = self.find_leaf_entry(key, name)

        if leaf is not None:
            k, v = leaf.keys[idx]
            return f"Знайдено: {v} (Хеш: {k})"
        return "Не знайдено"

    def search_range(self, name: str, condition=">"):
        key = self.hash_function(name)
        result = []

        if condition == ">":
            leaf = self.find_leaf(self.root, key)
            while leaf:
                for k, v in leaf.keys:
                    if k > key:
                        result.append(v)
                leaf = leaf.next_sibling

        elif condition == "<":
            leaf = self.get_first_leaf()
            while leaf:
                for k, v in leaf.keys:
                    if k < key:
                        result.append(v)
                    else:
                        return result
                leaf = leaf.next_sibling
        return result

    def find_leaf(self, node, key):
        if node.is_leaf:
            return node

        i = 0
        while i < len(node.keys) and key >= node.keys[i]:
            i += 1
        return self.find_leaf(node.children[i], key)

    def get_first_leaf(self):
        node = self.root
        while not node.is_leaf:
            node = node.children[0]
        return node

    def find_path_to_target_leaf(self, target_leaf):
        path = []

        def dfs(node):
            if node.is_leaf:
                return node is target_leaf

            for i, child in enumerate(node.children):
                path.append((node, i))
                if dfs(child):
                    return True
                path.pop()
            return False

        if dfs(self.root):
            return path
        return []

    def first_key_in_subtree(self, node):
        curr = node
        while not curr.is_leaf:
            curr = curr.children[0]
        if not curr.keys:
            return None
        return curr.keys[0][0]

    def rebuild_keys_from_children(self, node):
        if node.is_leaf:
            return

        node.keys = []
        for i in range(1, len(node.children)):
            sep = self.first_key_in_subtree(node.children[i])
            node.keys.append(sep if sep is not None else 0)

    def refresh_parent_keys(self, parent):
        self.rebuild_keys_from_children(parent)

    def rebalance_delete(self, path):
        min_leaf_keys = max(1, self.order // 2)
        min_internal_keys = max(1, self.order // 2 - 1)

        while path:
            parent, child_idx = path.pop()
            child = parent.children[child_idx]

            min_keys = min_leaf_keys if child.is_leaf else min_internal_keys
            if len(child.keys) >= min_keys:
                self.refresh_parent_keys(parent)
                continue

            left = parent.children[child_idx - 1] if child_idx > 0 else None
            right = parent.children[child_idx + 1] if child_idx < len(parent.children) - 1 else None

            if left is not None and len(left.keys) > min_keys:
                if child.is_leaf:
                    child.keys.insert(0, left.keys.pop())
                    parent.keys[child_idx - 1] = child.keys[0][0]
                else:
                    moved_child = left.children.pop()
                    child.children.insert(0, moved_child)
                    self.rebuild_keys_from_children(left)
                    self.rebuild_keys_from_children(child)
                    self.rebuild_keys_from_children(parent)
                continue

            if right is not None and len(right.keys) > min_keys:
                if child.is_leaf:
                    child.keys.append(right.keys.pop(0))
                    parent.keys[child_idx] = right.keys[0][0]
                else:
                    moved_child = right.children.pop(0)
                    child.children.append(moved_child)
                    self.rebuild_keys_from_children(right)
                    self.rebuild_keys_from_children(child)
                    self.rebuild_keys_from_children(parent)
                continue

            if left is not None:
                if child.is_leaf:
                    left.keys.extend(child.keys)
                    left.next_sibling = child.next_sibling
                else:
                    left.children.extend(child.children)
                    self.rebuild_keys_from_children(left)

                parent.children.pop(child_idx)
                self.rebuild_keys_from_children(parent)

            elif right is not None:
                if child.is_leaf:
                    child.keys.extend(right.keys)
                    child.next_sibling = right.next_sibling
                else:
                    child.children.extend(right.children)
                    self.rebuild_keys_from_children(child)

                parent.children.pop(child_idx + 1)
                self.rebuild_keys_from_children(parent)

        while (not self.root.is_leaf) and len(self.root.children) == 1:
            self.root = self.root.children[0]

        if not self.root.is_leaf:
            self.rebuild_keys_from_children(self.root)

    def delete(self, name: str):
        key = self.hash_function(name)
        leaf, idx = self.find_leaf_entry(key, name)

        if leaf is None:
            return False

        leaf.keys.pop(idx)

        if leaf is self.root:
            return True

        path = self.find_path_to_target_leaf(leaf)
        self.rebalance_delete(path)
        return True

    def visualize_tree(self):
        if self.root is None:
            return "Дерево порожнє"

        lines = []
        queue = [(self.root, 0)]
        current_level = 0
        level_nodes = []

        while queue:
            node, level = queue.pop(0)

            if level != current_level:
                lines.append(f"Рівень {current_level + 1}: " + " | ".join(level_nodes))
                level_nodes = []
                current_level = level

            if node.is_leaf:
                node_view = "[" + ", ".join(f"{name}:{key}" for key, name in node.keys) + "]"
            else:
                node_view = "<" + ", ".join(str(k) for k in node.keys) + ">"
                for child in node.children:
                    queue.append((child, level + 1))

            level_nodes.append(node_view)

        if level_nodes:
            lines.append(f"Рівень {current_level + 1}: " + " | ".join(level_nodes))

        leaf_chain = []
        first_leaf  = self.get_first_leaf()
        while first_leaf is not None:
            leaf_chain.append("[" + ", ".join(name for _, name in first_leaf.keys) + "]")
            first_leaf = first_leaf.next_sibling

        lines.append("Ланцюжок листків: " + " -> ".join(leaf_chain))
        return chr(10).join(lines)

In [7]:
from random import random

tree = BPlusTree(order=4)

names = [
    "Агада", "Андрух", "Білик", "Бондар", "Бойко", "Василенко", "Ващенко", "Гаврилюк",
    "Гнатюк", "Гончар", "Данилюк", "Демчук", "Жук", "Заварзін", "Заєць", "Зайченко",
    "Зайчатко", "Іваненко", "Калита", "Коваль", "Ковальчук", "Козак", "Козаченко", "Костенко",
    "Кравець", "Крамар", "Кулик", "Курча", "Левченко", "Литвин", "Макаренко", "Мельник",
    "Мороз", "Назаренко", "Олійник", "Паламарчук", "Петренко", "Романюк", "Савчук", "Сидоренко",
    "Ткаченко", "Федоренко", "Харченко", "Цимбал", "Чорний", "Шевченко", "Щербак", "Юрченко", "Ярема", "Яценко"
]

inserted_names = []
for name in names:
    try:
        tree.insert(name)
        inserted_names.append(name)
    except OverflowError as err:
        print(err)
        break

print(f"До вставки: {len(names)}")
print(f"Вставлено: {len(inserted_names)}")

existing = inserted_names[int(random() * len(inserted_names))]

print()
print("Точковий пошук:")
print(existing, "-", tree.search_exact(existing))
print("Неіснуючий -", tree.search_exact("Неіснуючий"))

print()
print("Діапазонний пошук відносно:", existing)
greater = tree.search_range(existing, ">")
less = tree.search_range(existing, "<")
print(f"Більше за '{existing}': {len(greater)} елементів")
print("Перші 10:", greater[:10])
print(f"Менше за '{existing}': {len(less)} елементів")
print("Перші 10:", less[:10])

print()
print("Перевірка видалення:")
targets_to_delete = [inserted_names[1], inserted_names[len(inserted_names) // 2], inserted_names[-1]]
for target in targets_to_delete:
    deleted = tree.delete(target)
    print(f"Видалення '{target}':", deleted, "| Пошук після видалення:", tree.search_exact(target))

print()
print("Візуалізація дерева:")
print(tree.visualize_tree())


До вставки: 50
Вставлено: 50

Точковий пошук:
Данилюк - Знайдено: Данилюк (Хеш: 321162459507)
Неіснуючий - Не знайдено

Діапазонний пошук відносно: Данилюк
Більше за 'Данилюк': 39 елементів
Перші 10: ['Демчук', 'Жук', 'Заварзін', 'Заєць', 'Зайчатко', 'Зайченко', 'Іваненко', 'Калита', 'Коваль', 'Ковальчук']
Менше за 'Данилюк': 10 елементів
Перші 10: ['Агада', 'Андрух', 'Білик', 'Бойко', 'Бондар', 'Василенко', 'Ващенко', 'Гаврилюк', 'Гнатюк', 'Гончар']

Перевірка видалення:
Видалення 'Андрух': True | Пошук після видалення: Не знайдено
Видалення 'Крамар': True | Пошук після видалення: Не знайдено
Видалення 'Яценко': True | Пошук після видалення: Не знайдено

Візуалізація дерева:
Рівень 1: <531154471006, 643372636508>
Рівень 2: <211191365607, 417353000003> | <536511238907, 611153163609> | <811165836508>
Рівень 3: <126362316006> | <226211795006, 321162459507> | <421134890005, 421152836508> | <536321159809, 536342183609> | <543321836508> | <616365640005, 635444564507> | <711121875006, 725311